# Vietnamese Patient Symptom Datasets — Full EDA (9 files)
This notebook inventories **five Parquet files, one CSV, three ZIP archives**. It runs the streaming EDA script from the `research/eda-all-patient-datasets` branch of the Pharmacy repository.

**Privacy:** Outputs contain aggregate statistics and hashed text fingerprints, not raw patient narratives. Results are corpus descriptions, not disease prevalence.


In [ ]:
%pip -q install pandas pyarrow matplotlib numpy openpyxl


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Put all 9 uploaded files in one Drive folder, preserving filenames.
INPUT_DIR = '/content/drive/MyDrive/PatientSymptomDatasets'
OUTPUT_DIR = '/content/drive/MyDrive/PatientSymptomEDA_Results'


In [ ]:
from pathlib import Path
EXPECTED = ['train-00000-of-00001.parquet', 'train.parquet',
            'train (1).parquet', 'train (2).parquet', 'train (3).parquet',
            'ViMedical_Disease.csv', 'Corpus_Redone.zip',
            'Question_for_dataset.zip', 'Corpus.zip']
root = Path(INPUT_DIR)
for f in EXPECTED:
    p = root/f
    print(f'{"OK" if p.is_file() else "MISSING":7}  {f:32}  {p.stat().st_size/1024**2:.2f} MiB' if p.is_file() else f'MISSING {f}')
assert all((root/f).is_file() for f in EXPECTED), 'Please place all 9 files in INPUT_DIR first.'


In [ ]:
from urllib.request import urlretrieve
from pathlib import Path
script_url = 'https://raw.githubusercontent.com/hungle2006/Pharmacy_app/research/eda-all-patient-datasets/research/eda_all_datasets.py'
local_script = '/content/eda_all_datasets.py'
urlretrieve(script_url, local_script)
print('Script downloaded:', Path(local_script).stat().st_size, 'bytes')


In [ ]:
import subprocess
cmd=['python', local_script, '--input-dir', INPUT_DIR, '--output-dir', OUTPUT_DIR]
result=subprocess.run(cmd, text=True, capture_output=True)
print(result.stdout[-15000:])
if result.returncode:
    print(result.stderr[-12000:])
    raise RuntimeError('EDA failed. See error traceback above.')


In [ ]:
import pandas as pd
from IPython.display import display, Markdown, Image
out=Path(OUTPUT_DIR)
display(Markdown((out/'overall_report.md').read_text(encoding='utf-8')))
for name in ['source_metrics.csv','cross_source_overlap.csv','column_completeness.csv','disease_labels.csv','symptom_keyword_heuristics.csv','zip_members.csv','errors.csv']:
    print('\n###',name)
    frame=pd.read_csv(out/name)
    display(frame.head(25))
for name in ['source_record_counts.png','duplicate_rates.png','top_symptom_keyword_mentions.png']:
    if (out/name).exists(): display(Image(filename=str(out/name)))


In [ ]:
# Package final report tables and PNG charts, excluding the hashed SQLite working index.
import zipfile
zip_out='/content/Patient_Symptom_9Datasets_EDA_Results.zip'
with zipfile.ZipFile(zip_out,'w',zipfile.ZIP_DEFLATED) as z:
    for p in out.glob('*'):
        if p.is_file() and p.suffix.lower() in ('.csv','.md','.png'):
            z.write(p,p.name)
print('Report package:',zip_out)
from google.colab import files
files.download(zip_out)
